In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from torchvision import datasets, transforms
from torch.nn import functional as F
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
import random
import numpy as np
seed=42
batch_size=128
epoch=100
lr=0.05
k=5
loss=nn.CrossEntropyLoss()
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 让CUDA尽可能使用确定性算法
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False


set_seed(seed)
def try_gpu(i=0):
    if torch.cuda.device_count()>=i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
device=try_gpu()

In [ ]:
def data_maker():
    mean = (
        0.4914,
        0.4822,
        0.4465
    )

    std = (
        0.2470,
        0.2435,
        0.2616
    )

    train_transform = transforms.Compose([
        # 填充成40×40，再随机裁剪回32×32
        transforms.RandomCrop(
            size=32,
            padding=4
        ),

        # 50%概率水平翻转
        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        # 轻微改变颜色
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),

        # 测试集必须使用同样的标准化参数
        transforms.Normalize(
            mean=mean,
            std=std
        )
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=True,
        transform=train_transform,
        download=False
    )
    valid_dataset = torchvision.datasets.CIFAR10(
            root="../cifar10-data",
            train=True,
            transform=test_transform,
            download=False
        )

    test_dataset = torchvision.datasets.CIFAR10(
        root="../cifar10-data",
        train=False,
        transform=test_transform,
        download=False
    )

    return train_dataset,valid_dataset,test_dataset

In [ ]:
def data_loader(dataset,batch_size,flag):
    return data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=flag,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )


In [ ]:
class normal_preactblock(nn.Module):
    def __init__(self,num):
        super().__init__()
        self.net=nn.Sequential()
        self.net.append(nn.BatchNorm2d(num))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(num,num,kernel_size=3,stride=1,padding=1))
        self.net.append(nn.BatchNorm2d(num))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(num,num,kernel_size=3,stride=1,padding=1))
    def forward(self,x):
        y=self.net(x)+x
        return y
class preactblock(nn.Module):
    def __init__(self,num):
        super().__init__()
        self.net1=nn.Sequential()
        self.net1.append(nn.BatchNorm2d(num))
        self.net1.append(nn.ReLU())
        self.net=nn.Sequential()
        self.net.append(nn.Conv2d(num,num*2,kernel_size=3,stride=2,padding=1))
        self.net.append(nn.BatchNorm2d(num*2))
        self.net.append(nn.ReLU())
        self.net.append(nn.Conv2d(num*2,num*2,kernel_size=3,stride=1,padding=1))
        self.nett=nn.Conv2d(num,num*2,kernel_size=1,stride=2)
    def forward(self,x):
        u=self.net1(x)
        y=self.net(u)+self.nett(u)
        return y


In [ ]:
def net_maker():
    b1=nn.Sequential(normal_preactblock(64),normal_preactblock(64))
    b2=nn.Sequential(preactblock(64),normal_preactblock(128))
    b3=nn.Sequential(preactblock(128),normal_preactblock(256))
    b4=nn.Sequential(preactblock(256),normal_preactblock(512))
    net=nn.Sequential(nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1),b1,b2,b3,b4,nn.BatchNorm2d(512),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(512,10))
    return net

In [ ]:
train_dataset,valid_dataset,test_dataset=data_maker()
def get_k_fold_indices(k,train_dataset,valid_dataset=None,seed=42):
    if valid_dataset==None:
        valid_dataset=train_dataset
    label=np.array(train_dataset.targets)
    k_fold=StratifiedKFold(n_splits=k,shuffle=True,random_state=seed)
    fold_indices = list(
    k_fold.split(np.zeros(len(label)),label)
    )
    return fold_indices
fold_indices=get_k_fold_indices(k,train_dataset,valid_dataset)
def get_k_fold_data(fold_indices,i,train_dataset,valid_dataset):
    train_indices,valid_indices=fold_indices[i]
    train_data=Subset(train_dataset,train_indices.tolist())
    valid_data=Subset(valid_dataset,valid_indices.tolist())
    return train_data,valid_data
def init(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(
            m.weight,
            mode="fan_out",
            nonlinearity="relu"  #表示后面接relu
        )

    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)


In [ ]:
for i in range(k):#先跑一折看看如何
    net=net_maker().to(device)
    train_data,valid_data=get_k_fold_data(fold_indices,i,train_dataset,valid_dataset)
    net.apply(init)
    train_iter=data_loader(train_data,batch_size,True)
    valid_iter=data_loader(valid_data,batch_size,False)
    trainer=torch.optim.SGD(net.parameters(),lr=lr,weight_decay=1e-3,momentum=0.9)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
    for j in range(epoch):
        net.train()
        train_loss=0.0
        train_count=0
        train_acc=0
        for x,y in train_iter:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            trainer.zero_grad()
            logit=net(x)
            l=loss(logit,y)
            train_loss+=l.item()*len(y)
            train_count+=len(y)
            train_acc+=(logit.argmax(dim=1)==y).sum().item()
            l.backward()
            trainer.step()
        scheduler.step()
        print(f"第{i+1}折的第{j+1}个epoch的训练loss：{train_loss/train_count:.5f},准确率:{train_acc/train_count:.5f}")
        net.eval()
        with torch.no_grad():
            valid_loss=0.0
            valid_count=0
            valid_acc=0.0
            for x,y in valid_iter:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                logit=net(x)
                l=loss(logit,y)
                valid_loss+=l.item()*len(y)
                valid_count+=len(y)
                valid_acc+=(logit.argmax(dim=1)==y).sum().item()
            print(f"第{i+1}折的第{j+1}个epoch的验证loss：{valid_loss/valid_count},准确率:{valid_acc/valid_count}")